# CW-DETR · Colab quickstart

A DINOv3 multi-task perception model for ADAS, built on RF-DETR.

This notebook lets you:
1. Clone the repo and install dependencies
2. Run the **shape sanity tests** (no model weights, no downloads — runs on CPU)
3. Build the full multi-task model with a **dummy backbone** and inspect outputs + parameter counts
4. *(Optional)* Build the **real DINOv3 backbone** after a Hugging Face login
5. Export the perception graph to **ONNX**

> Tip: `Runtime → Change runtime type → GPU` for the real-backbone cell. The dummy-backbone cells run fine on CPU.

## 0. Clone the repository

In [ ]:
# @title Clone CW-DETR
REPO_URL = "https://github.com/pirazor/CW-DETR.git"  # @param {type:"string"}
import os
if not os.path.isdir("CW-DETR"):
    !git clone --depth 1 $REPO_URL CW-DETR
%cd CW-DETR
!ls

## 1. Install dependencies
Colab already ships PyTorch + torchvision; we add the rest.

In [ ]:
!pip install -q transformers>=4.53.0 timm einops supervision scipy pyyaml onnx onnxslim
import torch, torchvision
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

## 2. Shape sanity tests (no weights / no download)
Validates the whole forward path — deformable attention, decoder, all five heads, the matcher and the multi-task loss — using a dummy backbone.

In [ ]:
!python -m tests.test_forward --config configs/cwdetr_nano_orin.yaml

## 3. Build the full model (dummy backbone) and inspect it
We mock the gated DINOv3 backbone with the test's `DummyBackbone`, so this runs anywhere.

In [ ]:
from unittest import mock
import torch
from cwdetr.config import load_config
from tests.test_forward import DummyBackbone

cfg = load_config("configs/cwdetr_nano_orin.yaml")
with mock.patch("cwdetr.models.cwdetr.DINOv3Backbone", DummyBackbone):
    from cwdetr.models.cwdetr import build_cwdetr
    model = build_cwdetr(cfg).eval()

img = torch.randn(1, 3, cfg.input.height, cfg.input.width)
with torch.no_grad():
    out = model(img)

print("detection logits :", tuple(out['detection']['pred_logits'].shape))
print("detection boxes  :", tuple(out['detection']['pred_boxes'].shape))
if 'segmentation' in out:
    print("drivable mask    :", tuple(out['segmentation']['drivable_logits'].shape))
    print("lane mask        :", tuple(out['segmentation']['lane_logits'].shape))

def count(m):
    return sum(p.numel() for p in m.parameters()) / 1e6
print("\nparameters (M):")
for name in ['projector', 'transformer', 'detection_head', 'seg_head', 'sign_head', 'traj_head']:
    mod = getattr(model, name, None)
    if mod is not None:
        print(f"  {name:16s} {count(mod):6.2f}M")

## 4. (Optional) Real DINOv3 backbone
DINOv3 weights are **gated** on the Hugging Face Hub. Run the login, accept the license for
[`facebook/dinov3-convnext-tiny-pretrain-lvd1689m`](https://huggingface.co/facebook/dinov3-convnext-tiny-pretrain-lvd1689m),
then build the real model. A GPU runtime is recommended.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
from cwdetr import load_config, build_cwdetr

cfg = load_config("configs/cwdetr_nano_orin.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_cwdetr(cfg).to(device).eval()   # downloads DINOv3 ConvNeXt-Tiny on first run

img = torch.randn(1, 3, cfg.input.height, cfg.input.width, device=device)
with torch.no_grad():
    out = model(img)
print("real-backbone forward OK — detection logits:", tuple(out['detection']['pred_logits'].shape))

## 5. Export the perception graph to ONNX
The deformable attention is `grid_sample`-based (ONNX opset ≥ 16), so it exports without a custom plugin — this is why CW-DETR deploys to Jetson with the stock TensorRT ONNX parser.

In [ ]:
# Export with the dummy backbone so it runs without gated weights.
from unittest import mock
import torch
from cwdetr.config import load_config
from cwdetr.export.export_onnx import CWDETRInfer
from tests.test_forward import DummyBackbone

cfg = load_config("configs/cwdetr_nano_orin.yaml")
with mock.patch("cwdetr.models.cwdetr.DINOv3Backbone", DummyBackbone):
    from cwdetr.models.cwdetr import build_cwdetr
    wrapper = CWDETRInfer(build_cwdetr(cfg).eval()).eval()

dummy = torch.randn(1, 3, cfg.input.height, cfg.input.width)
torch.onnx.export(wrapper, dummy, "cwdetr_nano.onnx",
                  input_names=["image"], output_names=["scores", "boxes", "drivable", "lane"],
                  opset_version=17, do_constant_folding=True,
                  dynamic_axes={"image": {0: "batch"}})
print("exported -> cwdetr_nano.onnx")
!ls -lh cwdetr_nano.onnx

---
**Next steps:** train with `cwdetr.engine.train` on BDD100K / nuScenes / GTSRB, then build a
TensorRT engine on a Jetson Orin with `cwdetr.export.build_tensorrt`. See
`docs/CW-DETR_Architecture_and_Strategy.md` for the full plan.